<a href="https://colab.research.google.com/github/beyzasakrakdil/ai-chemeng-learning/blob/main/02_Scaling_and_Visualization_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Scaling and Visualization

**Author:** Beyza ŞAKRAKDİL
**Based on:** AI in Chemical Engineering: Unlocking the Power Within Data (Romagnoli et al.)

This notebook covers:
- Why feature scaling is critical for process data
- Three important scalers: StandardScaler, MinMaxScaler, RobustScaler
- How to apply them correctly
- Basic exploratory visualization with Matplotlib and Seaborn




## 0. Import Libraries

We import the libraries we will need throughout this notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scalers from scikit-learn
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

import warnings
warnings.filterwarnings('ignore')   # hide unnecessary warnings

# Make plots look cleaner
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)


## 1. Generate Clean Synthetic Process Data

In the previous notebook we cleaned a dirty dataset.  
Here we generate a **clean** version of similar process data so we can focus only on scaling and visualization.

We create 300 samples of four process variables:
- `temperature` (°C)
- `pressure` (bar)
- `flow_rate`
- `concentration`


In [ ]:
# Set seed for reproducibility (same random numbers every time you run)
np.random.seed(42)
n = 300   # number of samples

# Create the DataFrame
df = pd.DataFrame({
    'temperature':   np.random.normal(loc=85, scale=8, size=n),   # mean=85, std=8
    'pressure':      np.random.normal(loc=2.5, scale=0.4, size=n),
    'flow_rate':     np.random.normal(loc=120, scale=15, size=n),
    'concentration': np.random.normal(loc=0.45, scale=0.08, size=n)
})

print("Shape of the data:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nStatistical summary:")
display(df.describe().T.round(3))


## 2. Why Do We Need Scaling?

Look at the ranges of our variables:

| Variable       | Typical range     |
|----------------|-------------------|
| temperature    | ~60 – 110         |
| pressure       | ~1.5 – 3.5        |
| flow_rate      | ~80 – 160         |
| concentration  | ~0.2 – 0.7        |

These variables have **very different scales**.  

Many machine learning algorithms calculate distances between points (KMeans, PCA, t-SNE, DBSCAN, etc.).  
If one variable has much larger numbers, it will dominate the distance calculation and the algorithm will almost ignore the other variables.

**Scaling** brings all features to a comparable range so that each variable contributes more fairly.


## 3. StandardScaler

StandardScaler is the most commonly used scaler.

It transforms each column so that:
- New mean ≈ 0
- New standard deviation ≈ 1

**Formula:**

$$
z = \frac{x - \mu}{\sigma}
$$

where  
- $x$ = original value  
- $\mu$ = mean of the column  
- $\sigma$ = standard deviation of the column

**When to use:**  
Most of the time (especially before PCA, KMeans, neural networks, etc.)


In [ ]:
# Select only the numeric columns we want to scale
numeric_cols = ['temperature', 'pressure', 'flow_rate', 'concentration']

# 1. Create the scaler object
scaler_standard = StandardScaler()

# 2. Make a copy of the original data (so we don't overwrite it)
df_standard = df.copy()

# 3. Fit the scaler on the data AND transform it
#    fit  → learns the mean and std of each column
#    transform → applies the formula to every value
df_standard[numeric_cols] = scaler_standard.fit_transform(df[numeric_cols])

print("After StandardScaler:")
print("Mean should be ~0 and std should be ~1\n")
display(df_standard[numeric_cols].describe().T.round(3))


## 4. MinMaxScaler

MinMaxScaler scales every feature to a fixed range.  
By default this range is **[0, 1]**.

**Formula:**

$$
x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

**When to use:**  
- When you need values strictly between 0 and 1  
- Neural networks sometimes prefer this  
- When the distribution is not Gaussian


In [ ]:
# Create MinMaxScaler
scaler_minmax = MinMaxScaler()

# Copy the original data
df_minmax = df.copy()

# Fit + transform
df_minmax[numeric_cols] = scaler_minmax.fit_transform(df[numeric_cols])

print("After MinMaxScaler:")
print("All values should now be between 0 and 1\n")
display(df_minmax[numeric_cols].describe().T.round(3))


## 5. RobustScaler

RobustScaler is designed to be **resistant to outliers**.

Instead of using mean and standard deviation, it uses:
- **Median** (middle value)
- **IQR** (Interquartile Range = 75th percentile – 25th percentile)

**Formula:**

$$
x_{scaled} = \frac{x - \text{median}}{\text{IQR}}
$$

**When to use:**  
When your data contains outliers and you do not want them to strongly affect the scaling.


In [ ]:
# Create RobustScaler
scaler_robust = RobustScaler()

# Copy the original data
df_robust = df.copy()

# Fit + transform
df_robust[numeric_cols] = scaler_robust.fit_transform(df[numeric_cols])

print("After RobustScaler:")
print("Median of each column should be close to 0\n")
display(df_robust[numeric_cols].describe().T.round(3))


## 6. Quick Comparison of the Three Scalers

| Scaler          | Uses                  | Sensitive to outliers? | Typical range after scaling |
|-----------------|-----------------------|------------------------|-----------------------------|
| StandardScaler  | mean + std            | Yes                    | roughly -3 to +3            |
| MinMaxScaler    | min + max             | Yes                    | 0 to 1                      |
| RobustScaler    | median + IQR          | No (more robust)       | varies                      |

For most chemical process datasets, **StandardScaler** is a safe default choice.


## 7. Basic Visualization

Before applying machine learning, it is very useful to **look at the data**.

We will create:
- Histograms (distribution of each variable)
- Boxplot
- Scatter plot
- Pairplot
- Correlation heatmap


### 7.1 Histograms – Distribution of each variable


In [ ]:
# Create a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Temperature
axes[0, 0].hist(df['temperature'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Temperature Distribution')
axes[0, 0].set_xlabel('Temperature (°C)')

# Pressure
axes[0, 1].hist(df['pressure'], bins=20, color='salmon', edgecolor='black')
axes[0, 1].set_title('Pressure Distribution')
axes[0, 1].set_xlabel('Pressure (bar)')

# Flow rate
axes[1, 0].hist(df['flow_rate'], bins=20, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Flow Rate Distribution')
axes[1, 0].set_xlabel('Flow Rate')

# Concentration
axes[1, 1].hist(df['concentration'], bins=20, color='orchid', edgecolor='black')
axes[1, 1].set_title('Concentration Distribution')
axes[1, 1].set_xlabel('Concentration')

plt.tight_layout()   # prevents overlapping titles/labels
plt.show()


### 7.2 Seaborn Histogram with KDE

KDE (Kernel Density Estimate) draws a smooth curve on top of the histogram.  
It helps us see the shape of the distribution more clearly.


In [ ]:
sns.histplot(data=df, x='temperature', bins=20, kde=True, color='steelblue')
plt.title('Temperature Distribution (with KDE curve)')
plt.xlabel('Temperature (°C)')
plt.show()


### 7.3 Boxplot

A boxplot shows:
- The median (line inside the box)
- The interquartile range (the box itself)
- Possible outliers (points outside the whiskers)


In [ ]:
sns.boxplot(data=df, y='temperature', color='lightblue')
plt.title('Temperature Boxplot')
plt.ylabel('Temperature (°C)')
plt.show()


### 7.4 Scatter Plot

Scatter plots are useful to see relationships between two variables.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['temperature'], df['pressure'], alpha=0.6, edgecolor='k', linewidth=0.3)
plt.xlabel('Temperature (°C)')
plt.ylabel('Pressure (bar)')
plt.title('Temperature vs Pressure')
plt.grid(True, alpha=0.3)
plt.show()


### 7.5 Pairplot

Pairplot shows the relationship between **every pair** of variables at once.  
It is one of the most useful plots for exploratory data analysis.


In [ ]:
sns.pairplot(df[numeric_cols], diag_kind='hist', corner=True)
plt.suptitle('Pairplot of Process Variables', y=1.02)
plt.show()


### 7.6 Correlation Heatmap

The correlation matrix tells us how strongly variables move together.
- +1 → perfect positive correlation
-  0 → no linear correlation
- -1 → perfect negative correlation


In [ ]:
# Calculate correlation matrix
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr,
            annot=True,          # write the numbers inside the cells
            cmap='coolwarm',     # blue → negative, red → positive
            center=0,            # 0 will be white
            fmt='.2f',           # 2 decimal places
            linewidths=0.5)

plt.title('Correlation Matrix of Process Variables')
plt.show()


## Summary

In this notebook we:

1. Generated clean synthetic process data
2. Learned **why** scaling is necessary
3. Applied three different scalers:
   - `StandardScaler` (most common)
   - `MinMaxScaler` (scales to 0-1)
   - `RobustScaler` (better with outliers)
4. Created several useful visualizations to understand the data

**Next notebook:** Dimensionality Reduction with PCA.
